## Build a semantic search engine with LangChain

we will build a search engine over a PDF document. This will allow us to retrieve passages in the PDF that are similar to an input query. The guide also includes a minimal RAG implementation on top of the search engine.


### Concepts
we will focuses on retrieval of text data. We will cover the following concepts 

- Documents and document loaders
- Text splitters
- Embeddings
- Vector stores and retrievers.

In [ ]:
!pip install langchain-community pypdf

## 1. Documents and document loaders 

Attributes are : - page_content , metadata,id(optional)

`metadata` attribute can capture information about the source of the document, 

In [ ]:
# generate sample documents (text containers with metadata)
from langchain_core.documents import Document
docs = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

### Loading documents 

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "../example-data/nke-10k-2023.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))


`PyPDFLoader` loads one    `Document` object per PDF page. For each, we can easily access: 
- The string contect of the page
- Metadata containing the file name and page number. 

In [ ]:
print(f"{docs[1].page_content[:200]}\n")
print(docs[1].metadata)

### Splitting 


##### Text Splitter (in Langchain)**
Text splitting breaks large documents into small , manageable chunks before embeddings 

#####  Why It's Needed
LLMs and embedding models have token limits. 
Smaller chunks -> better embeddings -> more accurate search results 

##### What it Does 
Takes big texts splits into overlapping pieces 

```
1000-token chunk
+ 200-token overlap (current chunk will have 200 token of previous chunk)
```

Overlap keeps context between chunks.(Overlap = context safety net)

##### When You Use It
After loading documents, before embeddings.
`Loader -> Text Splitter -> Embeddings -> Vector DB `

**Business Impact** : 
Improves answer quality, speeds retrieval, lowers cost.


(`In short Text splitter = cuts long text into AI-friendly chunks so search stays precise.`)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

print(len(all_splits))
# print(all_splits[0].page_content[:200])

## 2. Embeddings 
The idea is to store numeric vectors that are associated with the text. Given a query, we can embed it as a vector of the same dimension and use vector similarity metrics (such as cosine similarity) to identify related text.


In [ ]:
# installing huggingface langchain module for embeddings
!pip install  langchain-huggingface
# !pip install -qu langchain-google-genai

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
# no api key needed ( just local model download )
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])

## 3. Vector Stores
storing vector in vector db

In [ ]:
# install chroma db for vector store
!pip install  langchain-chroma

In [ ]:
from langchain_chroma import Chroma

# instantiated vector store , now we can index the documents
vector_store = Chroma(
    collection_name="pdf_docs_collection",
    embedding_function=embeddings,
    persist_directory="../local_chromaDb/",  # Where to save data locally, remove if not necessary
)

In [ ]:
ids = vector_store.add_documents(documents=all_splits)

# Embeddings typically represent text as a “dense” vector such that texts with similar meanings are geometrically close

In [ ]:
# Embeddings typically represent text as a “dense” vector such that texts with similar meanings are geometrically close

# return documents based on similarity to a string query 
results = vector_store.similarity_search(
    "How many distribution centers does Nike have in the US?"
)

print(results[0])

Async query : 

In [ ]:
# Async query 


# (non-blocking,means it can run concurrently with other operations)
# usefull when you need to perform multiple searches or other async operations without waiting for each one to complet sequentially 
results = await vector_store.asimilarity_search("When was Nike incorporated?")

print(results[0])

Return scores : 

In [ ]:
# Note that providers implement different scores; the score here
# is a distance metric that varies inversely with similarity.

results = vector_store.similarity_search_with_score("What was Nike's revenue in 2023?")
results2 = vector_store.similarity_search_with_score("My name is Aakash and day by day in every way I am getting better and better.")
doc, score = results[0]
doc2, score2 = results2[0]
print(f"Similar one's Score: {score}\n")
print(f"Unsimilar one's Score: {score2}\n")
# print(doc)

Return documents based on similarity to an embedded query:

In [ ]:
# in tihs we are directly providing the embedding vector instead of a text query
embedding = embeddings.embed_query("How were Nike's margins impacted in 2023?")

results = vector_store.similarity_search_by_vector(embedding) 
print(results[0])

#intergrations specific docs -  https://docs.langchain.com/oss/python/integrations/vectorstores

## Retriever (in LangChain)

A **retriever** is a small component whose job is to:

➡️ Take a user question  
➡️ Search the vector database  
➡️ Return the most relevant documents  

It does **not** generate answers. It only finds information.

---

### Why Retrievers Exist
Separates *searching* from *answering*.
This makes RAG systems modular and scalable.

---

### What below code does 

- Wraps `similarity_search()` into a reusable callable  
- Accepts a query string  
- Returns top matching `Document` objects  

---

### Where It Fits in RAG
User Question  
↓  
Retriever  
↓  
Relevant Documents   
↓   
LLM   
↓   
Final Answer  




Retriever = **Smart search function for your knowledge base**

In [ ]:
from typing import List 
from langchain_core.documents import Document
from langchain_core.runnables import chain

@chain
def retriever(query: str)-> List[Document]:
    return vector_store.similarity_search(query,k=1)

retriever.batch(
    [
        "When was Nike founded? ",
        "Who is Nike's CEO?",
    ],
)